## Exercise 1: Set up the ClearCover Insurance Data Platform

Before the data quality pipeline can run, the Unity Catalog structure and raw source data must be in place.

Run each cell in this notebook **from top to bottom**. After setup is complete, return to the lab instructions page and continue with Exercise 2.

### Task 1: Create the Unity Catalog structure

This cell is provided for you — run it to prepare your environment.

It creates the following Unity Catalog objects:
- A **catalog** named `insurance_lab` — the top-level namespace for the ClearCover Insurance platform
- A **bronze schema** inside `insurance_lab` — raw claims data, exactly as it arrives from source systems
- A **silver schema** inside `insurance_lab` — validated and type-safe claims
- A **gold schema** inside `insurance_lab` — aggregated reporting data
- A **volume** named `raw_files` inside `insurance_lab.bronze` — landing zone for incoming CSV claim files

In [ ]:
# Since no default storage is enabled, we are inheriting the storage path from the default catalog's root.
# Use the current catalog to reliably find the workspace default catalog,
# regardless of its naming convention.
default_catalog = spark.catalog.currentCatalog()

storage_root = (
    spark.sql(f"DESCRIBE CATALOG EXTENDED {default_catalog}")
    .filter("info_name = 'Storage Root'")
    .select("info_value")
    .first()[0]
)
print (f"Storage root: {storage_root}")

spark.sql(f"""
    CREATE CATALOG IF NOT EXISTS insurance_lab
    MANAGED LOCATION '{storage_root}'
    COMMENT 'ClearCover Insurance analytics platform'
""")

In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS insurance_lab.bronze
  COMMENT 'Raw claims data, unmodified as received from source systems';

CREATE SCHEMA IF NOT EXISTS insurance_lab.silver
  COMMENT 'Validated claims, type-checked and quality-constrained';

CREATE SCHEMA IF NOT EXISTS insurance_lab.gold
  COMMENT 'Aggregated reporting data for dashboards';

CREATE VOLUME IF NOT EXISTS insurance_lab.bronze.raw_files
  COMMENT 'Landing zone for raw CSV claim files';

### Task 2: Write the raw claims CSV to the volume

The dataset contains **intentional quality issues**: missing identifiers, malformed dates, invalid amounts, and incorrect status values — the exact problems your pipeline will fix.

> `claim_date` and `claim_amount` are intentionally loaded as **STRING** columns. Type validation is the pipeline's job.

In [ ]:
# ✅ Run this cell — writes the raw claims CSV to the volume
volume_path = '/Volumes/insurance_lab/bronze/raw_files'

rows = [
    'claim_id,policy_id,customer_id,claim_date,incident_date,claim_amount,claim_type,status,coverage_amount,agent_code',
    'CLM-001,POL-1001,CUST-001,2026-01-05,2026-01-03,4500.00,AUTO,OPEN,25000.00,AGT-01',
    'CLM-002,POL-1002,CUST-002,2026-01-07,2026-01-06,12000.00,HOME,PENDING,150000.00,AGT-02',
    'CLM-003,POL-1003,CUST-003,2026-01-10,2026-01-09,250.00,HEALTH,CLOSED,5000.00,AGT-01',
    'CLM-004,,CUST-004,2026-01-12,2026-01-11,8900.00,AUTO,OPEN,30000.00,AGT-03',
    'CLM-005,POL-1005,CUST-005,2026-01-14,2026-01-13,,LIFE,PENDING,200000.00,AGT-02',
    ',POL-1006,CUST-006,2026-01-15,2026-01-14,3200.00,HOME,CLOSED,80000.00,AGT-04',
    'CLM-007,POL-1007,CUST-007,not-a-date,2026-01-16,7600.00,AUTO,OPEN,35000.00,AGT-03',
    'CLM-008,POL-1008,CUST-008,2026-01-18,2026-01-17,N/A,HEALTH,PENDING,10000.00,AGT-01',
    'CLM-009,POL-1009,CUST-009,2026-01-20,2026-01-19,-500.00,HOME,INVALID_STATUS,120000.00,AGT-04',
    'CLM-010,POL-1010,CUST-010,2026-01-22,2026-01-21,15000.00,VEHICLE,CLOSED,50000.00,AGT-02',
    'CLM-011,POL-1011,CUST-011,2026-01-25,2026-01-24,1800.00,AUTO,PENDING,20000.00,AGT-05',
    'CLM-012,POL-1012,CUST-012,2026-01-27,2026-01-26,6400.00,HEALTH,OPEN,15000.00,AGT-01',
    'CLM-013,POL-1013,CUST-013,2026-01-28,2026-01-27,92000.00,LIFE,CLOSED,300000.00,AGT-03',
    'CLM-014,POL-1014,CUST-014,2026-01-30,invalid-date,-900.00,AUTO,PENDING,22000.00,AGT-02',
    'CLM-015,POL-1015,CUST-015,2026-02-01,2026-01-31,3750.00,HOME,OPEN,95000.00,AGT-05',
    'CLM-016,POL-1016,CUST-016,2026-02-03,2026-02-02,550.00,HEALTH,CLOSED,8000.00,AGT-02',
    'CLM-017,POL-1017,CUST-017,2026-02-05,2026-02-04,18500.00,LIFE,PENDING,250000.00,AGT-03',
    'CLM-018,POL-1018,CUST-018,2026-02-07,2026-02-06,2200.00,AUTO,OPEN,28000.00,AGT-01',
    'CLM-019,POL-1019,,2026-02-09,2026-02-08,4100.00,HOME,PENDING,110000.00,AGT-04',
    'CLM-020,POL-1020,CUST-020,2026-02-11,2026-02-10,67000.00,LIFE,CLOSED,400000.00,AGT-03'
]

dbutils.fs.put(volume_path + '/insurance_claims_raw.csv', '\n'.join(rows) + '\n', overwrite=True)
print('✅ CSV written to', volume_path)

### Task 3: Load the CSV into the bronze Delta table

Note the explicit schema — `claim_date` and `claim_amount` are kept as **STRING** so the pipeline type-checking exercises work correctly.

In [ ]:
# ✅ Run this cell — loads the CSV into the bronze Delta table
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

schema = StructType([
    StructField('claim_id',        StringType(), True),
    StructField('policy_id',       StringType(), True),
    StructField('customer_id',     StringType(), True),
    StructField('claim_date',      StringType(), True),   # intentionally STRING
    StructField('incident_date',   StringType(), True),
    StructField('claim_amount',    StringType(), True),   # intentionally STRING
    StructField('claim_type',      StringType(), True),
    StructField('status',          StringType(), True),
    StructField('coverage_amount', DoubleType(), True),
    StructField('agent_code',      StringType(), True),
])

volume_path = '/Volumes/insurance_lab/bronze/raw_files'

df = (
    spark.read
    .option('header', 'true')
    .option('nullValue', '')
    .schema(schema)
    .csv(volume_path + '/insurance_claims_raw.csv')
)

(
    df.write
    .format('delta')
    .mode('overwrite')
    .saveAsTable('insurance_lab.bronze.claims_raw')
)

print('✅ Table insurance_lab.bronze.claims_raw created')
print(f'   Total rows loaded: {df.count()}')

### Verify: Inspect the raw data

Run the cells below to confirm the table was created successfully and preview the data.

Scan the results for rows with quality issues — these are the targets for your pipeline exercises.

In [ ]:
%sql
SELECT *
FROM insurance_lab.bronze.claims_raw
ORDER BY claim_id NULLS LAST;

In [ ]:
%sql
DESCRIBE TABLE insurance_lab.bronze.claims_raw;

---

✅ **Setup complete!** Return to the lab instructions page and continue with Exercise 2.

In [ ]:
%sql
-- Optional cleanup — uncomment to remove all lab resources when done
-- DROP CATALOG IF EXISTS insurance_lab CASCADE;